# Adım 4 — Chunking ve kontrol

1. hücre `data/clean/*.txt` ve `data/legal/documents.yaml` dosyalarından `data/chunks/chunks.jsonl` üretir.  
Sonraki hücreler çıktıyı kontrol eder. Beklenen sonuçlar her hücrenin başında yazıyor.

In [ ]:
from chunky.chunk import main

dropped = main()

In [ ]:
# Beklenen: her satırda "açıklanamayan eksik: []", tüm metadata alanları dolu, anahtar ifadeler True.
import json, re
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().parent
chunks = [json.loads(line) for line in (ROOT / "data" / "chunks" / "chunks.jsonl").open(encoding="utf-8")]
by_id = {c["chunk_id"]: c for c in chunks}

# 1) Madde numaralarındaki boşluklar yalnızca mülga olup atılan maddeler mi? (belge ve madde türü başına)
groups, repealed = defaultdict(list), defaultdict(set)
for c in chunks:
    if m := re.fullmatch(r"(m|ekm|gm)(\d+)[A-ZÇ]?", c["chunk_id"].split("-", 1)[1]):
        groups[(c["doc_code"], m[1])].append(int(m[2]))
for cid in dropped:
    code, part = cid.split("-", 1)
    if m := re.fullmatch(r"(m|ekm|gm)(\d+)[A-ZÇ]?", part):
        repealed[(code, m[1])].add(int(m[2]))
for (code, kind), nums in groups.items():
    missing = set(range(1, max(nums) + 1)) - set(nums)
    print(f"{code:8} {kind:4} {len(nums):4} madde  mülga: {len(missing & repealed[(code, kind)]):3}  "
          f"açıklanamayan eksik: {sorted(missing - repealed[(code, kind)])}")

# 2) Metadata eksiksiz mi? (article_heading ve section bazı chunk'larda boş olabilir)
required = ["chunk_id", "article_id", "doc_code", "doc_title", "doc_type", "article_label",
            "source_url", "accessed", "version_date", "text"]
empty = [(c["chunk_id"], f) for c in chunks for f in required if not c.get(f)]
print("\nBoş zorunlu alan:", empty[:10] or "yok")

# 3) Tablo ve ek içerikleri doğru chunk'ta mı?
for cid, phrase in [("KTY-m100", "Otomobil (M1), (M1G), | 50 | 90 | 110 | 120"),
                    ("SAĞLIK-m10", "Turnike"), ("KTY-cetvel1", "eflektif")]:
    print(cid, phrase in by_id[cid]["text"])

In [ ]:
# Eval setindeki her atıf bir chunk'a karşılık geliyor mu?  Beklenen: "eşleşmeyen: 0"
import openpyxl

EVAL_PATH = ROOT / "docs" / "eval" / "Eval_Set_1.xlsx"   # dosya başka yerdeyse burayı değiştir
#C:\projects\chunky-502\docs\eval\Eval_Set_1.xlsx
REF = re.compile(r"(KTK|KTY|KURS|SAĞLIK|MUAYENE|SKUTER)\s+(Art\.|Additional Article|Provisional Article|Annex-?)\s*(\d+)(/[A-ZÇ](?![\w-]))?")
KIND = {"Art.": "m", "Additional Article": "ekm", "Provisional Article": "gm"}

def to_chunk_id(code, kind, no, letter):
    if kind.startswith("Annex"):                       # KTY'de ekler "N Sayılı Cetvel" adını taşır
        return next((f"{code}-{p}{no}" for p in ("cetvel", "ek", "liste") if f"{code}-{p}{no}" in by_id), f"{code}-ek{no}")
    return f"{code}-{KIND[kind]}{no}{(letter or '').strip('/')}"

ws = openpyxl.load_workbook(EVAL_PATH, read_only=True)["Questions"]
rows = list(ws.iter_rows(values_only=True))
col = rows[0].index("Legal Article Reference")
total, unmatched = 0, []
for row in rows[1:]:
    ref = row[col] or ""
    for m in REF.findall(ref):
        cid = to_chunk_id(*m)
        total += 1
        if cid not in by_id:
            unmatched.append((row[0], cid))
print(f"{total} atıf, eşleşmeyen: {len(unmatched)}", unmatched)
print("Eşleşmeyenlerden mülga olup atılanlar:", [u for u in unmatched if u[1] in dropped])

In [ ]:
# Bilgi: en uzun chunk'lar (alt-parçalama kararı embedding modeli seçilince verilecek, AD-08)
for c in sorted(chunks, key=lambda c: -len(c["text"]))[:10]:
    print(f'{c["chunk_id"]:16} {len(c["text"]):6} karakter  {c["article_label"]}')
print("\nToplam:", len(chunks), "chunk")

In [ ]:
# Değişiklik notları temizlendi mi?
# Beklenen: kalan tarihli parantez 5 (KTY-gm10'da 2 içerik + KURS-m7, m24, m28'de Danıştay notu),
# kalıntılar 0, yalnız '(…)' 2 (KTK-m65, KTK-m90; bilerek bırakıldı), kontrollerin hepsi True.
dated = re.compile(r"\([^()]*\d{1,2}/\d{1,2}/\d{4}[^()]*\)")
left = [(c["chunk_id"], p) for c in chunks for p in dated.findall(c["text"])]
print("Kalan tarihli parantez:", len(left))
for cid, p in left:
    print("  ", cid, p[:90])

for mark in ["md.)", "RG-", "(…)", "(...)"]:
    hits = [c["chunk_id"] for c in chunks if mark in c["text"]]
    print(f"{mark!r:8} kalıntı: {len(hits)} {hits[:10]}")

print("\nNot sayısı (amendments):", sum(len(c["amendments"]) for c in chunks))
print(f"Mülga olup atılan ({len(dropped)}):", dropped)

def has_line(cid, line):
    return line in by_id[cid]["text"].split("\n")

for ok, name in [
    (not has_line("KTK-m3", "Minibüs:"), "KTK-m3: boş 'Minibüs:' satırı yok"),
    (not has_line("KURS-m15", "(2)"), "KURS-m15: boş '(2)' satırı yok"),
    ("eğitim ve sınav aracı," in by_id["KURS-m17"]["text"], "KURS-m17: noktalamadan önce boşluk yok"),
    ("Bir eğitim döneminde" in by_id["KURS-m16"]["text"], "KURS-m16: not çıkınca cümle düzgün"),
    ("Jandarma Genel Komutanlığı ile Sahil Güvenlik" in by_id["KTK-m22"]["text"], "KTK-m22: yeniden düzenlenen bent korundu"),
    (all(cid not in by_id for cid in ["KTK-m5A", "KTY-m51A", "KURS-gm5", "SAĞLIK-ek1"]), "tamamen mülga maddeler/ekler atıldı"),
    (by_id["KTK-m1"]["version_date"] == "2026-07-31", "KTK sürüm tarihi 2026-07-31"),
]:
    print(ok, " ", name)

In [ ]:
from chunky import chunk as ch
import yaml

for d in yaml.safe_load(ch.DOCS_FILE.read_text(encoding="utf-8"))["documents"]:
    lines = (ch.CLEAN_DIR / f"{d['file']}.txt").read_text(encoding="utf-8").splitlines()
    units = ch.find_units(lines)
    for k, u in enumerate(units):
        cid = f"{d['doc_code']}-{u['part']}"
        end = units[k + 1]["start"] if k + 1 < len(units) else len(lines)
        if cid in dropped:
            print(cid, "|", " / ".join(lines[u["start"]:end])[:160])
        for line in lines[u["start"]:end]:
            if "(…)" in line and cid in ("KTK-m65", "KTK-m90"):
                print("   (…) satırı →", cid, "|", line[:200])

In [ ]:
import json
import re
from pathlib import Path

# Repo kökünü bul: pyproject.toml'un olduğu klasöre kadar yukarı çık
kok = Path.cwd()
while not (kok / "pyproject.toml").exists():
    if kok.parent == kok:
        raise FileNotFoundError("pyproject.toml bulunamadı")
    kok = kok.parent

with open(kok / "data" / "chunks" / "chunks.jsonl", encoding="utf-8") as f:
    chunk_listesi = [json.loads(satir) for satir in f]

print(len(chunk_listesi), "chunk okundu")

In [ ]:
def goster(i, en_fazla=3000):
    c = chunk_listesi[i]
    print(f"[{i}] {c['chunk_id']}  |  {c['doc_title']} – {c['article_label']} {c['article_heading']}")
    print("Bölüm:", c["section"])
    print("Uzunluk:", len(c["text"]), "karakter")
    print("-" * 80)
    metin = c["text"]
    if en_fazla and len(metin) > en_fazla:
        metin = metin[:en_fazla] + f"\n… (kesildi, {len(c['text']) - en_fazla} karakter daha var)"
    print(metin)
    print("=" * 80, "\n")

def bul(chunk_id):
    for i, c in enumerate(chunk_listesi):
        if c["chunk_id"] == chunk_id:
            return i
    raise KeyError(f"Bulunamadı: {chunk_id}")

# İndekse göre:
#for i in [0, 1, 50, 200, 400]:
#    goster(i)

# Kimliğe göre (istersen):
#goster(bul("KTK-m51"))

In [ ]:
goster(10)

In [ ]:
from collections import Counter, defaultdict

not_kalibi = re.compile(r"\(([^()]*\d{1,2}/\d{1,2}/\d{4}[^()]*)\)")
on_ek = re.compile(
    r"^\s*(?:(?:ek |geçici )?madde\s+[\w/]+\s*[–‒-]\s*)?(?:\(\d+\)\s*)?(?:[a-zçğıöşü]{1,2}\)\s*)?",
    re.IGNORECASE,
)

turler = Counter()
konumlar = defaultdict(Counter)
ornekler = {}
etiketsiz = []

for c in chunk_listesi:
    for satir in c["text"].split("\n"):
        govde = satir[on_ek.match(satir).end():]
        for m in not_kalibi.finditer(satir):
            parantez = m.group(0)
            icerik = m.group(1)
            tur = icerik.split(":", 1)[0].strip() if ":" in icerik else ""
            if not tur or re.search(r"\d", tur) or len(tur) > 50:
                etiketsiz.append((c["chunk_id"], parantez))
                continue
            kalan = satir.replace(parantez, "")
            if not kalan[on_ek.match(kalan).end():].strip():
                konum = "tek başına"
            elif govde.startswith(parantez):
                konum = "satır başı"
            else:
                konum = "satır içi"
            turler[tur] += 1
            konumlar[tur][konum] += 1
            ornekler.setdefault(tur, (c["chunk_id"], parantez))

print(f"{'Tür':<40}{'Adet':>6}   satır başı / satır içi / tek başına")
print("-" * 90)
for tur, adet in turler.most_common():
    k = konumlar[tur]
    print(f"{tur:<40}{adet:>6}   {k['satır başı']:>10} / {k['satır içi']:>9} / {k['tek başına']:>10}")

print("\nHer türden bir örnek:")
for tur, _ in turler.most_common():
    cid, ornek = ornekler[tur]
    print(f"- {tur} → {cid}: {ornek[:150]}")

print(f"\nEtiketsiz tarihli parantez: {len(etiketsiz)} (ilk 25):")
for cid, p in etiketsiz[:25]:
    print(f"- {cid}: {p[:150]}")

In [ ]:
for c in chunk_listesi:
    for satir in c["text"].split("\n"):
        for m in not_kalibi.finditer(satir):
            etiket = m.group(1).split(":", 1)[0].lower()
            if "mülga" in etiket or "iptal" in etiket:
                kalan = satir.replace(m.group(0), "")
                if kalan[on_ek.match(kalan).end():].strip():
                    print(f"{c['chunk_id']}: {satir[:200]}")